# Appendix 04: Matrix Properties and Decompositions

Source orientation: printed pages 578-587; PDF pages 596-605.

This notebook is an original, standalone computational treatment of the chapter. The PDF was used only to identify the chapter structure, concepts, and algorithmic emphasis. It does not reproduce textbook prose, figures, screenshots, long exercise text, or page crops. The goal is to turn the chapter into an inspectable multiple-view-geometry lab that a reader can use without keeping the book open.

## Chapter Goal

Rank, nullspaces, eigenstructure, SVD, Cholesky, QR/RQ, Schur, and matrix identities used by MVG algorithms.

Multiple-view geometry becomes easier to learn when every algebraic object is paired with a scene, a measurement, and a diagnostic. This notebook therefore treats the chapter as a working vision problem rather than as a sequence of isolated formulas. Points, lines, cameras, conics, tensors, residuals, and optimization variables are represented in executable form. The visuals are designed to reveal what survives a projection, what is lost, which constraints are merely algebraic, and which constraints are geometric.

The examples use deterministic synthetic data: calibrated and uncalibrated cameras, planar grids, cube or room-like point sets, noisy correspondences, and small track matrices. Synthetic data is intentional because every artifact can be regenerated, perturbed, and checked. Real images are valuable in applications, but the central ideas of this chapter are clearest when the ground truth geometry is known and the failure modes can be turned on deliberately.

## Translation Guide

- **Homogeneous data:** points, lines, cameras, planes, and tensors are represented up to scale, so every equality that involves them must be phrased as a proportionality, an incidence relation, a rank condition, or a normalized residual.
- **Camera action:** a camera is a projective map from 3D scene coordinates to 2D image coordinates. The code always distinguishes the camera center, the image measurement, and the back-projected ray or plane that connects them.
- **Invariant:** the important question is not whether an array changed, but whether the geometric relation survived: collinearity, coplanarity, cross-ratio, rank, epipolar incidence, positive depth, or reprojection error.
- **Estimator:** a linear algorithm supplies an initial model; geometric, statistical, or robust criteria decide whether that model explains the measurements.
- **Artifact:** each figure is saved under the book-local `artifacts/` tree, displayed inline, and checked in the final cell so the notebook remains reproducible.

## Route Through The Chapter

1. Name the geometric object and its computational representation.
2. Build a small scene where the object can be projected, transformed, or estimated.
3. Draw the construction in a way that makes the invariant visible.
4. Perturb the data to expose conditioning, uncertainty, or ambiguity.
5. Close with checks that assert the core relations and artifact integrity.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = 'appendix-04'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


## Visual Storyboard And Library Routing

This appendix is the linear-algebra toolbox behind cameras, epipoles, homographies, and tensor estimates. **NumPy** and **SciPy RQ** are used for the actual decompositions so the code matches the algorithms used elsewhere in the course. **Matplotlib** provides durable matrix heatmaps, singular-value panels, and geometric action diagrams. The artifacts live under `artifacts/appendix-04` and each check is tied to a matrix property used by MVG algorithms.

| Visual | Artifact | What to inspect | Check |
| --- | --- | --- | --- |
| Orthogonal action on vectors | `figures/orthogonal-action-preserves-norms.png` | rotations preserve lengths and oriented area | `Q.T @ Q = I`, determinant near `+1` |
| RQ camera decomposition | `figures/rq-camera-decomposition-heatmaps.png` | camera left block splits into upper-triangular calibration and rotation | reconstructed matrix error |
| SVD rank and nullspace | `figures/svd-rank-nullspace-diagnostics.png` | small singular values expose constraints and null vectors | null residual is small |
| Skew matrix cross product | `figures/skew-cross-product-nullspace.png` | `[a]_x b` equals `a x b`, and `a` is the null vector of `[a]_x` | cross-product and nullspace residuals |

## Core Concepts

### 1. Matrix decompositions expose geometric structure hidden in coefficients

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **SVD geometry panel**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **SVD reconstruction matches the original matrix**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 2. Svd identifies rank, nullspaces, conditioning, and best low-rank approximations

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **nullspace and rank dashboard**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **nullspace vectors satisfy A v near zero**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 3. Rq-style decompositions support camera calibration factorization

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **RQ camera factorization sketch**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **positive-definite matrices have positive eigenvalues**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 4. Positive definite matrices encode valid covariance and conic constraints

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **positive-definite ellipse view**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **factorized camera matrices reconstruct P up to scale**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

## Worked Example Pattern

The worked examples use a shared synthetic lab. A few cameras view a simple 3D scene, those cameras produce image measurements, and the chapter-specific object is computed from the measurements. This pattern is repeated because it makes the course cumulative: homographies from Part 0 return as plane-induced transfers in Part II, camera matrices from Part I feed epipolar geometry, and two-view triangulation becomes a building block for N-view bundle adjustment.

For this chapter, the important work is to connect **Rank, nullspaces, eigenstructure, SVD, Cholesky, QR/RQ, Schur, and matrix identities used by MVG algorithms.** to observable behavior. The first figure is a concept map that states what to watch. The second figure is a geometry scene. The third figure is a diagnostic view where residuals, conditioning, or covariance can be inspected. The fourth figure is a rank or constraint dashboard, because many multiple-view failures announce themselves as a singular value that should not be ignored.

The code is intentionally compact. It is not a production vision library; it is a transparent teaching implementation that exposes each step. The reusable helpers live in `utils/` so later chapters can use the same projection, epipolar, estimation, and plotting vocabulary.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import rq

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib
from utils.cameras import camera_matrix, look_at_rotation, make_calibration, skew

ENTRY_TITLE = 'Matrix Properties and Decompositions'
MODE = 'matrix-decompositions'
TOPIC = 'appendix-04'
CONCEPTS = ['orthogonal matrices', 'RQ decomposition', 'SVD rank and nullspace', 'skew-symmetric cross-product matrices', 'least-squares constraints']
VISUALS = ['orthogonal action preserves norms', 'RQ camera decomposition heatmaps', 'SVD rank/nullspace diagnostics', 'skew cross-product nullspace']
CHECKS = ['orthogonal matrix preserves vector norms', 'RQ factors reconstruct the camera block', 'small singular vector satisfies the constraint matrix', 'skew matrix implements cross product and has expected nullspace']
SEED = 704
rng = np.random.default_rng(SEED)
artifact_paths = []

def rot_z(theta):
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

def rot_y(theta):
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])

Q = rot_z(math.radians(34.0)) @ rot_y(math.radians(-18.0))
vectors = rng.normal(size=(8, 3))
rotated = (Q @ vectors.T).T
norm_error = float(np.max(np.abs(np.linalg.norm(vectors, axis=1) - np.linalg.norm(rotated, axis=1))))
K = make_calibration(820.0, 790.0, 310.0, 235.0, skew_value=12.0)
C = np.array([1.2, -0.4, -4.6])
R = look_at_rotation(C, target=np.array([0.0, 0.1, 3.1]))
P = camera_matrix(K, R, C)
M = P[:, :3]
K_rq, R_rq = rq(M)
signs = np.sign(np.diag(K_rq)); signs[signs == 0] = 1.0
T_sign = np.diag(signs)
K_rq = K_rq @ T_sign
R_rq = T_sign @ R_rq
if np.linalg.det(R_rq) < 0:
    K_rq[:, 0] *= -1
    R_rq[0, :] *= -1
K_rq = K_rq / K_rq[2, 2]
M_reconstructed = K_rq @ R_rq
rq_scale = float(np.vdot(M, M_reconstructed) / np.vdot(M_reconstructed, M_reconstructed))
rq_error = float(np.linalg.norm(M - rq_scale * M_reconstructed) / np.linalg.norm(M))
A = rng.normal(size=(8, 4))
true_null = np.array([0.7, -0.2, 0.3, 1.0])
A[:, -1] = -(A[:, :3] @ true_null[:3]) / true_null[-1]
U, S, Vt = np.linalg.svd(A)
null_vec = Vt[-1]
null_residual = float(np.linalg.norm(A @ null_vec))
a = np.array([0.8, -1.1, 0.45])
b = np.array([-0.2, 0.55, 1.3])
Sx = skew(a)
cross_error = float(np.linalg.norm(Sx @ b - np.cross(a, b)))
skew_null_error = float(np.linalg.norm(Sx @ a))

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.8))
ax.scatter(vectors[:, 0], vectors[:, 1], s=42, color='#2563eb', label='original xy components')
ax.scatter(rotated[:, 0], rotated[:, 1], s=42, color='#c2410c', marker='^', label='after orthogonal Q')
for p, q in zip(vectors, rotated):
    ax.plot([p[0], q[0]], [p[1], q[1]], color='#94a3b8', lw=0.8)
unit = np.column_stack([np.cos(np.linspace(0, 2*np.pi, 240)), np.sin(np.linspace(0, 2*np.pi, 240))])
ax.plot(unit[:, 0], unit[:, 1], color='#0f766e', lw=1.4, label='unit circle reference')
ax.set_title('Orthogonal matrices preserve lengths while rotating coordinates', loc='left', fontsize=12, fontweight='bold')
ax.set_aspect('equal', adjustable='box'); ax.grid(True, color='#e5e7eb'); ax.legend(fontsize=8)
orthogonal_path = save_matplotlib(fig, TOPIC, 'figures', 'orthogonal-action-preserves-norms.png')
plt.close(fig)
artifact_paths.append(orthogonal_path)
display_artifact(orthogonal_path, width=720)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.4))
for ax, mat, title in [(axes[0], M, 'camera block M'), (axes[1], K_rq, 'upper-triangular K'), (axes[2], R_rq, 'orthogonal R')]:
    im = ax.imshow(mat, cmap='coolwarm')
    ax.set_title(title, loc='left', fontsize=10, fontweight='bold')
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    for (i, j), value in np.ndenumerate(mat):
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle('RQ decomposition is the camera-calibration factorization used by finite cameras', x=0.02, ha='left', fontsize=12, fontweight='bold')
fig.tight_layout()
rq_path = save_matplotlib(fig, TOPIC, 'figures', 'rq-camera-decomposition-heatmaps.png')
plt.close(fig)
artifact_paths.append(rq_path)
display_artifact(rq_path, width=880)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.9))
axes[0].semilogy(np.arange(1, len(S)+1), S, marker='o', color='#6d28d9', lw=2.2)
axes[0].set_title('Singular values reveal a one-dimensional nullspace', loc='left', fontsize=10, fontweight='bold')
axes[0].set_xlabel('index'); axes[0].set_ylabel('singular value'); axes[0].grid(True, color='#e5e7eb')
axes[1].bar(np.arange(len(null_vec)), null_vec / np.max(np.abs(null_vec)), color='#2563eb')
axes[1].set_title('Recovered homogeneous null vector', loc='left', fontsize=10, fontweight='bold')
axes[1].set_xticks(range(4), ['h0', 'h1', 'h2', 'h3']); axes[1].grid(True, axis='y', color='#e5e7eb')
fig.tight_layout()
svd_path = save_matplotlib(fig, TOPIC, 'figures', 'svd-rank-nullspace-diagnostics.png')
plt.close(fig)
artifact_paths.append(svd_path)
display_artifact(svd_path, width=840)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.0))
im = axes[0].imshow(Sx, cmap='coolwarm')
axes[0].set_title('[a]_x skew matrix', loc='left', fontsize=10, fontweight='bold')
for (i, j), value in np.ndenumerate(Sx):
    axes[0].text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
labels = ['[a]x b', 'np.cross(a,b)', '[a]x a']
vals = [np.linalg.norm(Sx @ b), np.linalg.norm(np.cross(a, b)), np.linalg.norm(Sx @ a)]
axes[1].bar(labels, vals, color=['#2563eb', '#0f766e', '#c2410c'])
axes[1].set_title('Cross product action and null vector', loc='left', fontsize=10, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20); axes[1].grid(True, axis='y', color='#e5e7eb')
fig.tight_layout()
skew_path = save_matplotlib(fig, TOPIC, 'figures', 'skew-cross-product-nullspace.png')
plt.close(fig)
artifact_paths.append(skew_path)
display_artifact(skew_path, width=820)

## Computational Lab

The lab below uses the same checks as the visual storyboard:

- `SVD reconstruction matches the original matrix`
- `nullspace vectors satisfy A v near zero`
- `positive-definite matrices have positive eigenvalues`
- `factorized camera matrices reconstruct P up to scale`

The exact values are less important than the workflow. Build a synthetic configuration, compute the projective or statistical object, and verify the invariant that the chapter says should hold. In a real application the data would be image correspondences or tracked features. In this course the data is deterministic so the result can be audited.

The miniature experiment uses two cameras and a cube-like point cloud whenever possible. Even chapters about statistics, tensor notation, or optimization keep the same projective heartbeat: measurements are generated by projection, a model is estimated, and the model is judged by residuals, rank, or covariance. This continuity helps prevent a common misconception in multiple-view geometry: that the algebra and the geometry are separate topics. They are two views of the same constraints.

In [ ]:
matrix_checks = {
    'source_span': {'printed_pages': '578-587', 'pdf_pages': '596-605'},
    'libraries': ['numpy linear algebra', 'scipy.linalg.rq', 'matplotlib matrix/action diagnostics', 'course artifact helpers'],
    'orthogonality_error': float(np.linalg.norm(Q.T @ Q - np.eye(3))),
    'rotation_determinant': float(np.linalg.det(Q)),
    'norm_preservation_error': norm_error,
    'rq_reconstruction_error': rq_error,
    'rq_rotation_orthogonality_error': float(np.linalg.norm(R_rq.T @ R_rq - np.eye(3))),
    'svd_smallest_singular_value': float(S[-1]),
    'nullspace_residual': null_residual,
    'skew_cross_product_error': cross_error,
    'skew_null_residual': skew_null_error,
    'artifact_count': len(artifact_paths),
}
checks_path = save_json(matrix_checks, TOPIC, 'checks', 'matrix-decomposition-invariants.json')
display_artifact(checks_path)
matrix_checks

## Pitfalls And Failure Modes

The main danger in this chapter is to confuse a plausible array with a valid geometric object. A matrix can have the right shape and still violate rank, scale, sign, or incidence constraints. A small algebraic residual can hide a large image-plane error if the coordinate system is poorly normalized. A projective reconstruction can reproject perfectly while still being metrically wrong. A calibration estimate can look numerically precise while being driven by a degenerate camera motion or by points that do not constrain the intended degrees of freedom.

The antidote is to make each assumption executable. When a relation is homogeneous, normalize before comparing. When a model is estimated, inspect both the residual distribution and the singular values. When an upgrade is claimed, state which additional object was fixed: the line at infinity, the plane at infinity, the absolute conic, the absolute dual quadric, or a cheirality condition. When a robust method is used, report inliers and outliers instead of only the final model. These habits keep the notebook honest and make the visualizations diagnostic rather than decorative.

## Applied Lab

Replace the synthetic data in the lab with one of your own small image-measurement sets. Keep the same artifact contract:

1. draw the measurements and the estimated relation;
2. save the figure under `artifacts/appendix-04/figures/`;
3. write a JSON summary under `artifacts/appendix-04/checks/`;
4. assert the invariant that matters for the chapter.

For this notebook, a good extension is to perturb the measurements with three noise levels and compare the resulting diagnostics. Watch whether **SVD reconstruction matches the original matrix** degrades smoothly or fails abruptly. Abrupt failures usually indicate rank loss, degeneracy, a poor parameterization, or an unhandled scale convention.

In [ ]:
final_sanity = {
    'artifact_count': len(artifact_paths),
    'all_artifacts': [str(path.relative_to(BOOK_ROOT)) for path in artifact_paths],
    'check_artifact': str(checks_path.relative_to(BOOK_ROOT)),
    'orthogonality_error': matrix_checks['orthogonality_error'],
    'rq_reconstruction_error': matrix_checks['rq_reconstruction_error'],
    'nullspace_residual': matrix_checks['nullspace_residual'],
    'skew_cross_product_error': matrix_checks['skew_cross_product_error'],
}
assert_artifacts(artifact_paths, min_bytes=1500)
assert checks_path.exists() and checks_path.stat().st_size > 200
assert final_sanity['artifact_count'] >= 4
assert matrix_checks['orthogonality_error'] < 1e-12
assert abs(matrix_checks['rotation_determinant'] - 1.0) < 1e-12
assert matrix_checks['norm_preservation_error'] < 1e-12
assert matrix_checks['rq_reconstruction_error'] < 1e-8
assert matrix_checks['rq_rotation_orthogonality_error'] < 1e-12
assert matrix_checks['nullspace_residual'] < 1e-10
assert matrix_checks['skew_cross_product_error'] < 1e-12
assert matrix_checks['skew_null_residual'] < 1e-12
final_path = save_json(final_sanity, TOPIC, 'checks', 'final-sanity.json')
final_sanity

## Takeaways

- Matrix decompositions expose geometric structure hidden in coefficients.
- Svd identifies rank, nullspaces, conditioning, and best low-rank approximations.
- Rq-style decompositions support camera calibration factorization.
- Positive definite matrices encode valid covariance and conic constraints.

The chapter's durable lesson is that multiple-view geometry is a discipline of representations and invariants. The visual artifacts show the representation; the code checks the invariant; the prose explains why the invariant matters. That triangle is what makes the notebook stand alone from the source text.